<a href="https://colab.research.google.com/github/collinboler/cos432A6/blob/main/Copy_of_A6_PatchGuard_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PatchGuard: A Robust, Adaptive Defense for Image Classifiers

## Meta Info

To get start, make a copy of this notebook to your own Google Drive, then work on this assignment in that notebook. Make sure to share your notebook (Viewer access to everyone) when you finished it, and include the link to it below so we can access it.
```
Group members: [TODO]

Link to your Colab Notebook: [TODO]
```

## Introduction

In this assignment, you'll learn about how image classifiers can be attacked and how they can then be made more robust against attacks.
Specifically, you will learn about **patch-based attacks** which are inference time attacks that can be realized in the physical world.
You will then explore how **redundancy-based defenses** like PatchGuard can protect against such attacks and how they can be practically deployed in real-world contexts.

To learn more about this defense, also known as PatchGuard, you can read more about it [here](https://www.usenix.org/system/files/sec21-xiang.pdf) and its follow-up work, PatchCleanser, [here](https://arxiv.org/pdf/2108.09135).

As in previous assignments, we will provide helper code and clearly indicate which functions you need to implement using the following annotation:

```
# TODO: write your code here
```
Moreover, there is a set of reflection questions in part 6 that you must answer. All required text responses will be denoted by **WRITE YOUR ANSWER HERE**.


This assignment does not assume you must have taken prior coursework or have in-depth experience with machine learning.
To focus on the core concepts of the assignment, we abstract away  low-level details through pre-implemented helper functions that allow you to interact with the machine learning models (e.g., during training and testing).


## 0. Getting Started

Make a copy of this notebook so you can edit the assignment directly.

Change the Hardware to GPU by going to Runtime > Change Runtime Type > Hardware Accelerator and set it to T4 GPU.

**IMPORTANT:** Although there is nothing to implement or answer in Section 0 and its subsections, read the subsection descriptions carefully to familiarize yourself with the assignment specifications.

### 0.1. Install Libraries
Run the following block of code to install the required libraries.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

### 0.2. Load Datasets
Run the following block of code to load the MNIST training and test datasets.

**MNIST** is a dataset of 28×28 pixel grayscale images of hand-written digits (0-9), where each image is labeled by the digit it represents.

There is nothing to implement or answer in this step.


In [ ]:
device = "cuda"
epochs_baseline = 30
epochs_pg = 100
n_examples_train = 3000
n_examples_test = 500
data_dir = "./data"
batch_size=10

def load_subset(dataset, n, device):
    xs = torch.stack([dataset[i][0] for i in range(n)])
    ys = torch.tensor([dataset[i][1] for i in range(n)])
    return xs.to(device), ys.to(device)


# load MNIST data
train = datasets.MNIST(root=data_dir, train=True, download=True, transform=transforms.ToTensor())
test = datasets.MNIST(root=data_dir, train=False, download=True, transform=transforms.ToTensor())

train_images, train_labels = load_subset(train, n_examples_train, device)
test_images_clean, test_labels = load_subset(test, n_examples_test, device)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 500kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.59MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.85MB/s]


### 0.2. Setup Patch Attack & PatchGuard Parameters
Run the following block of code to setup the patch attack and PatchGuard parameters. In the reflection questions, you will be asked to explore different parameters.

There is nothing to implement or answer in this step.


In [ ]:
# -----------------------------------------------------------------------------
# Patch attack parameters
# -----------------------------------------------------------------------------

# Size (in pixels) of the adversarial patch (size x size)
patch_sizes = [12,]

# Step size (in pixels) when sliding the patch across the image
# Smaller stride = more overlap
strides = [2,]

# Size (in pixels) of the window (window_size x window_size) PatchGuard model predicts on
window_sizes = [16,]

# Offsets (in pixels) from the image center where the patch is placed
patch_x = [12,]
patch_y = [12,]

### 0.3. Prepare Baseline Models

Run the following blocks of code to prepare the **baseline (i.e., unprotected)** image classifiers for the MNIST dataset.

For **MNIST**, we train a small neural network based on the classic **LeNet** architecture.  
LeNet is one of the earliest neural networks designed for recognizing handwritten digits.

If you are interested, you can read more about it [here](https://www.geeksforgeeks.org/computer-vision/what-is-lenet/).

There is nothing to implement or answer in this step.



In [ ]:
# -----------------------------------------------------------------------------
# Basic CNN Image Classifier (based on the classic LeNet architecture)
#
# This neural network takes an image as input and predicts which class
# the image belongs to (e.g., which digit in MNIST).
# -----------------------------------------------------------------------------
class BasicCNN(nn.Module):
    def __init__(self, patch_size, in_channels, channels=(32, 64), fc_dim=128, num_classes=10):
        super().__init__()

        c1, c2 = channels
        feat_dim = c2 * (patch_size // 4) * (patch_size // 4)

        self.net = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(feat_dim, fc_dim),
            nn.ReLU(),
            nn.Linear(fc_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# -----------------------------------------------------------------------------
# Train a model on a dataset
#
# The model repeatedly looks at batches of training images and gradually
# improves its predictions by minimizing classification error.
# -----------------------------------------------------------------------------
def train_model(model, x, y, epochs):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    N = x.size(0)

    for epoch in range(epochs):
        perm = torch.randperm(N)
        epoch_loss = 0.0

        for i in range(0, N, batch_size):
            idx = perm[i:i + batch_size]

            inputs = x[idx].to(device)
            labels = y[idx].to(device)

            logits = model(inputs)
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch:02d} | "f"avg loss = {epoch_loss / (N // batch_size):.4f}")

In [ ]:

# -----------------------------------------------------------------------------
# Create and train baseline MNIST image classifier
# -----------------------------------------------------------------------------
baseline_model = BasicCNN(patch_size=train_images.size(-1), in_channels = train_images.size(1)).to(device)
train_model(baseline_model, train_images, train_labels, epochs_baseline)

### 0.4. Generate Adversarially-Perturbed Images

Run the following blocks of code to generate **adversarial images**.

An *adversarial image* is created by modifying a small region of an image so that a classifier makes an incorrect prediction.
In this assignment, the modification is a **k × k patch**, where the pixels inside the patch can take arbitrary values.

The code below automatically computes a patch that causes the model to misclassify each image.

In the original **PatchGuard** paper, the attacker is allowed to place the patch **anywhere in the image**. For simplicity in this assignment, we instead **fix the patch location** during image generation.

There is nothing to implement or answer in this step.


In [ ]:
# -----------------------------------------------------------------------------
# Generate Adversarial Images with Patch Attacks
#
# This function creates adversarial images by adding a small k×k patch
# to each input image. The patch is optimized so that the classifier
# is more likely to make an incorrect prediction.
#
# In the full PatchGuard threat model, an attacker can place the patch
# anywhere in the image. For simplicity in this assignment, we fix the
# patch location near the center of the image.
#
# The algorithm repeatedly adjusts the patch pixel values to increase
# the model's classification error on the patched image.
# -----------------------------------------------------------------------------
def generate_adversarial_img(model, x, y, patch_size, patch_x, patch_y, steps=20, step_size=8/255):
    patch_eps=1.0

    B, C, H, W = x.shape

    # Choose a fixed patch location (near the center of the image)
    r = H // 2 - patch_y
    c = W // 2 - patch_x
    patch_slice = (slice(None), slice(None), slice(r, r + patch_size), slice(c, c + patch_size))

    # Initialize patch with zeros
    patch = torch.zeros(B, C, patch_size, patch_size, device=x.device, requires_grad=True)
    # Optimize the patch only
    x_base = x.detach()

    # Iteratively update the patch to make the model misclassify the image
    for _ in range(steps):
        # Insert patch onto image
        patched = x_base.clone()
        patched[patch_slice] = (patched[patch_slice] + patch).clamp(0, 1)

        # Compute loss on patched image to maximize misclassification
        loss = F.cross_entropy(model(patched), y)
        loss.backward()

        # Update patch
        with torch.no_grad():
            patch.add_(step_size * patch.grad.sign()) # Gradient ascent
            patch.clamp_(-patch_eps, patch_eps) # Constrains patch magnitude for stable training
            patch.grad.zero_()

    # Save adversarial image + patch
    x_adv = x_base.clone()
    x_adv[patch_slice] = (x_adv[patch_slice] + patch).clamp(0, 1)

    return x_adv

# -----------------------------------------------------------------------------
# Generate adversarial images for MNIST
# -----------------------------------------------------------------------------
test_images_adv = generate_adversarial_img(baseline_model,
                                           test_images_clean,
                                           test_labels,
                                           patch_sizes[0],
                                           patch_x[0], patch_y[0])

## 1. A Motivating Example
### 1.2. The Effect of Benign vs Adversarial Inputs on Non-PatchGuard Image Classifiers

Before diving into the assignment, let us first examine a motivating example that demonstrates how **unmitigated image classifiers can fail when given adversarial inputs**.

There is nothing to implement or answer in this step.

First, consider how the MNIST classifier performs on a **benign (clean) input image**.
In this setting, the model receives an unmodified image from the dataset and typically predicts the correct label.


In [ ]:
# -----------------------------------------------------------------------------
# Visualize Clean MNIST Image (single plot)
# -----------------------------------------------------------------------------
def show_clean_image(model, img, label):
    model.eval()

    with torch.no_grad():
        pred = model(img.unsqueeze(0)).argmax(dim=1).item()

    img = img.detach().cpu()

    plt.figure(figsize=(3, 3))
    plt.imshow(img.squeeze(), cmap="gray")
    plt.title(f"MNIST\nTrue: {label}, Pred: {pred}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


# -------------------------------------------------------------------------
# Example clean image
# -------------------------------------------------------------------------
idx = 10
show_clean_image(baseline_model, test_images_clean[idx], test_labels[idx])

Now we compare this to an **adversarially perturbed version of the same image**.  
In this example, a small **k × k patch** has been inserted into the image. Although the patch only modifies a small region, it is carefully constructed to cause the classifier to make an incorrect prediction.

This example highlights an important security concern.
Even though a **human observer may still correctly recognize the image**, the classifier is easily manipulated by a small localized perturbation.

Such vulnerabilities can be dangerous in real-world systems.
For example, in autonomous driving, attackers can place a small sticker on a **stop sign** causing vision systems to misclassify it as a different object, potentially leading to unsafe behavior.

However, play around with the `idx` variable in the previous code block (and rerun both code blocks) to see other examples (e.g., `idx=7`) of adversarial perturbations that even we as humans may not be able to classify correctly.
Think about how this may also affect how we use "robust accuracy" as defined in part 5.

In [ ]:
# -----------------------------------------------------------------------------
# Visualize Adversarial MNIST Image (single plot)
# -----------------------------------------------------------------------------
def show_adversarial_image(model, img_adv, label):
    model.eval()

    with torch.no_grad():
        pred = model(img_adv.unsqueeze(0)).argmax(dim=1).item()

    img_adv = img_adv.detach().cpu()

    plt.figure(figsize=(3, 3))
    plt.imshow(img_adv.squeeze(), cmap="gray")
    plt.title(f"MNIST (Adversarial)\nTrue: {label}, Pred: {pred}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


# -------------------------------------------------------------------------
# Example adversarial image
# -------------------------------------------------------------------------
show_adversarial_image(baseline_model, test_images_adv[idx], test_labels[idx])

### 1.2. PatchGuard
We just observed how a small adversarial patch can break the classifier, causing it to produce incorrect predictions.
This type of vulnerability is exactly what **PatchGuard** is designed to defend against.

PatchGuard improves robustness to adversarial patches by making predictions based on **smaller regions of an image**, also known as a **window**, rather than processing the full image at once.
The PatchGuard process is as follows:
1. Divide the image into windows
    - the image is split into multiple (possibly overlapping) windows
2. Window-based classification
    - For each window, the model predicts the image class using only that window, with all other pixels zeroed out.
3. Robust aggregation
    - The predictions from all windows are combined (via majority voting) to produce a more robust final classification.

As a result, even if some windows are corrupted by the adversarial patch, the goal is that the correct predictions from unaffected windows will  dominate the final result.


## 2. Threat Model Questions

Please answer the questions below.

1. What are the attacker's goals?

WRITE YOUR ANSWER HERE


2. What is the attacker's background knowledge (e.g., about the model, training data)?

WRITE YOUR ANSWER HERE


3. What are the attacker's capabilities and in what ways are they constrained?

WRITE YOUR ANSWER HERE



## 3. Prepare PatchGuard Model

Next, we prepare **PatchGuard-protected version** of the MNIST classifier.

As a reminder, PatchGuard improves robustness to adversarial patches by making predictions on **windows**, rather than on the entire image at once. In particular the PatchGuard models are trained on images where all pixels outside of the window are zeroed out, whereas unmitigated models train on the full image context.

### Coding Task 1:
To implement this defense, you will complete the function **`extract_windows`**, which extracts all sliding windows from an input image. The PatchGuard model will later make predictions on each of these windows.

Your implementation should:

- Extract all windows of size **`window_size × window_size`** from the input image **`img`**
- Ensure the windows slide across the image using the specified **`stride`**
- Return the extracted windows in the format expected by the training code

Important PyTorch concepts for this assignment are tensors (multi-dimensional arrays) and common operations on them, such as accessing their dimensions using `.shape`. Here is PyTorch documentation on [tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html).
- Image data are typically represented as 3D tensors with shape C × H × W, where:
  - C = number of channels (e.g., 1 for MNIST which is greyscale)
  - W = width
  - H = height

Once you finish implementing this function, run the following blocks of code to **train the PatchGuard model** using these windows.

Training may take a few minutes.


In [ ]:
# -----------------------------------------------------------------------------
# Extract Sliding Windows from an Image
#
# Given an input image of shape [channels, height, width], this function
# extracts all sliding windows of size [window_size × window_size] using
# the specified stride.
#
# Each window represents a small region of the image that the PatchGuard
# classifier will later use for prediction.
#
# Returns:
#   Tensor of shape [num_windows, channels, window_size, window_size]
# -----------------------------------------------------------------------------
def extract_windows(img, window_size, stride):
    # TODO: write your code here


In [ ]:
# -----------------------------------------------------------------------------
# Train a PatchGuard Model
#
# Unlike the baseline classifier, PatchGuard does not train directly on
# full images. Instead, each image is broken into many sliding windows.
#
# The model is trained to classify each window individually. During
# inference, PatchGuard aggregates predictions across windows to
# mitigate the effect of adversarial patches.
# -----------------------------------------------------------------------------
def train_pg_model(model, x, y, epochs, window_size, stride=None):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    N = x.size(0)

    for epoch in range(epochs):
        perm = torch.randperm(N)
        epoch_loss = 0.0

        # For each image in the batch, extract its sliding windows
        for i in range(0, N, batch_size):
            idx = perm[i:i + batch_size]

            window_batches = []
            label_batches = []

            for j in idx:
                img = x[j]
                label = y[j].item()

                windows = extract_windows(img, window_size, stride)
                window_batches.append(windows)
                # Each window inherits the label of the original image
                label_batches.append(torch.full((windows.size(0),), label, device=windows.device))

                inputs = torch.cat(window_batches, dim=0).to(device)
                labels = torch.cat(label_batches, dim=0).to(device)

            logits = model(inputs)
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch:02d} | "f"avg loss = {epoch_loss / (N // batch_size):.4f}")

In [ ]:
# -----------------------------------------------------------------------------
# Train PatchGuard Models
# -----------------------------------------------------------------------------

print("MNIST training:")
patch_model = BasicCNN(patch_size=window_sizes[0], in_channels = train_images.size(1)).to(device)
train_pg_model(patch_model, train_images, train_labels, epochs_pg, window_sizes[0], strides[0])

## 4. Implement Prediction and Accuracy Calculation Functions

Now we implement the functions needed to make predictions using PatchGuard and evaluate its performance.

Recall that PatchGuard does not classify the entire image directly. Instead, it:

1. Extracts sliding windows from the image
2. Classifies each window individually
3. Aggregates window predictions using majority voting

This aggregation helps limit the influence of a small adversarial patch, since only a subset of windows may contain the perturbed patch.

### Coding Tasks 2 & 3

Implement the following functions:

- **`patchguard_predict`** which predicts the label of a single image using PatchGuard's window-based classification and majority voting.

- **`patchguard_accuracy`** which computes the overall classification accuracy of a PatchGuard model on a dataset.

In [ ]:
# helper function that takes in an array of windows and the model to classify each window
# returns an array of predictions (one per window)
def get_window_predictions(model, windows):
  with torch.no_grad():
    logits = model(windows)
    return logits.argmax(dim=1)

# -----------------------------------------------------------------------------
# PatchGuard Prediction Function
#
# Predicts the label of a single image using the PatchGuard defense.
#
# Returns:
#   Integer representing the predicted class label.
# -----------------------------------------------------------------------------
def patchguard_predict(model, img, window_size, stride):
  with torch.no_grad():
    # 1) extract all windows from image
    # TODO: write your code here


    # 2) classify each window using model and get each window's prediction
    # TODO: write your code here


    # 3) use majority voting over window predictions to get final label
    # TODO: write your code here


In [ ]:
# -----------------------------------------------------------------------------
# PatchGuard Accuracy Evaluation
#
# Computes the classification accuracy of a PatchGuard model over a dataset.
#
# For each image:
#   - Predict the label using patchguard_predict
#   - Compare it with the true label
#
# Returns:
#   Accuracy = (# correct predictions) / (# test images)
# -----------------------------------------------------------------------------
def patchguard_accuracy(model, x, y, window_size, stride):
    model.eval()
    correct = 0
    num_test_img = x.size(0)

    # TODO: write your code here
    for i in range(num_test_img):


    return correct / num_test_img

## 5. Evaluate Model on Benign and Adversarial Images

Finally, we evaluate how well the models perform on **benign (clean) images** and **adversarially perturbed images**.

For the MNIST dataset, we compute two metrics:

- **Clean accuracy** – accuracy on the original test images  
- **Robust accuracy** – accuracy on adversarially-perturbed test images

Comparing these metrics highlight how vulnerable unmitigated image classifiers are to adversarial patches and how robustness may inadverdently affect clean accuracy.

There is nothing to implement or answer in this step. Run the following code blocks to evaluate the models.

In [ ]:
# -----------------------------------------------------------------------------
# Compute Classification Accuracy
#
# Evaluates a standard image classifier on a dataset by comparing
# predicted labels with the true labels.
#
# Returns:
#   Accuracy = (# correct predictions) / (# test images)
# -----------------------------------------------------------------------------
def accuracy(model, x, y):
    model.eval()
    correct = 0
    num_test_img = x.size(0)

    for i in range(num_test_img):
        logits = model(x[i].unsqueeze(0))
        pred = logits.argmax(dim=1)

        label = y[i].item()
        correct += int(pred == label)
    return correct / num_test_img

In [ ]:
# -----------------------------------------------------------------------------
# Evaluate Baseline (Unmitigated) Model
# -----------------------------------------------------------------------------
print("Unmitigated:")
print("Clean accuracy:", accuracy(baseline_model, test_images_clean, test_labels))
print("Robust accuracy:", accuracy(baseline_model, test_images_adv, test_labels))

In [ ]:
# -----------------------------------------------------------------------------
# Evaluate PatchGuard Model
# -----------------------------------------------------------------------------
print("PatchGuard:")
acc_clean = patchguard_accuracy(patch_model, test_images_clean, test_labels, window_sizes[0], strides[0])
print(f"Clean accuracy:    {acc_clean:.3f}")
acc_adv = patchguard_accuracy(patch_model, test_images_adv, test_labels, window_sizes[0], strides[0])
print(f"Robust accuracy:   {acc_adv:.3f}")

## 6. Reflection Questions

Please answer the following questions. For each question, cite accuracy statistics for **5 different values** of the parameter, while holding other parameters constant. You can adjust the parameters in part 0.2.

1. How does the size of the **sliding window** affect the unmitigated and PatchGuard model accuracies? Test out different sizes in the code block below and run it to show your results.

WRITE YOUR ANSWER HERE


In [ ]:
# TODO: write your code here to test different window sizes to answer the questions below

2. How does the size of the **stride** affect the unmitigated and PatchGuard model accuracies? Test out different sizes in the code block below and run it to show your results.

WRITE YOUR ANSWER HERE


In [ ]:
# TODO: write your code here to test different strides to answer the questions below

3. In the paper, PatchGuard assumes that the attacker's patch is not fixed, but we do it here for simplicity when generating adversarial images. How does **adversarial patch location** affect the unmitigated and PatchGuard model accuracies? Test out different sizes in the code block below and run it to show your results.

WRITE YOUR ANSWER HERE


In [ ]:
# TODO: write your code here to test different adv. patch locations to answer the questions below

4. How does the size of the **adversarial patch size** affect the unmitigated and PatchGuard model accuracies? Test out different sizes in the code block below and run it to show your results.

WRITE YOUR ANSWER HERE


In [ ]:
# TODO: write your code here to test different adv. patch sizes to answer the questions below

## Submission Checklist

Write all the answers in this Colab notebook. Once you are finished, generate
1) a PDF via (File -> Print -> Save as PDF)
2) a .ipynb file (File -> Download .ipynb)

Compress all files (together with the PDF and .ipynb files from the other notebook) in one .zip file and submit it to Gradescope.

- **Important:** check your PDF and .ipynb files before you submit to Gradescope to make sure it exported correctly. For example, if Colab gets confused about your syntax, it will sometimes terminate the PDF creation routine early.
